# ORPO and SimPO: Reference-Free Alignment

DPO still requires a reference model in memory during training. ORPO and SimPO eliminate this requirement — they incorporate the preference signal directly into the training loss without needing to compare against a frozen reference policy.

## The Reference Model Bottleneck

DPO requires computing log probabilities under both the policy being trained and a frozen reference model. This means:
- Two forward passes per training step
- The reference model must be kept in memory (or recomputed from disk, which is slow)
- For a 7B model, this adds ~14GB to training memory

For large-scale training where memory is the constraint, eliminating the reference model is a meaningful improvement.

## ORPO: Odds Ratio Preference Optimization

ORPO (Hong et al. 2024) combines supervised fine-tuning and preference alignment into a single loss:

```
L_ORPO = L_SFT + λ * L_OR
```

Where L_SFT is the standard next-token prediction loss on chosen responses, and L_OR is an odds ratio term:

```
L_OR = -log σ(log[odds_θ(y_w|x) / odds_θ(y_l|x)])
```

The odds of a response: `odds(y|x) = P(y|x) / (1 - P(y|x))`

By training on the chosen response with SFT loss simultaneously, ORPO implicitly treats the current model state as the reference — penalizing rejected responses relative to the model's own current generation.

In [ ]:
import math

def log_odds(log_prob: float) -> float:
    prob = math.exp(log_prob)
    prob = max(1e-8, min(1 - 1e-8, prob))
    return math.log(prob / (1 - prob))

def orpo_loss(log_prob_chosen: float, log_prob_rejected: float,
              sft_loss: float, lam: float = 0.1) -> dict:
    lo_chosen = log_odds(log_prob_chosen)
    lo_rejected = log_odds(log_prob_rejected)
    log_ratio = lo_chosen - lo_rejected
    preference_loss = -math.log(1.0 / (1.0 + math.exp(-log_ratio)) + 1e-8)
    total = sft_loss + lam * preference_loss
    return {
        'sft_loss': round(sft_loss, 4),
        'preference_loss': round(preference_loss, 4),
        'total': round(total, 4),
        'odds_ratio': round(math.exp(log_ratio), 4),
    }

# Show how ORPO loss behaves across scenarios
print('ORPO Loss Analysis:')
print(f'{'Scenario':<30} {'SFT':>7} {'Pref':>7} {'Total':>7} {'Odds Ratio':>11}')
print('-' * 65)
scenarios = [
    ('Good chosen, bad rejected', -1.0, -4.0, 1.5),
    ('Good chosen, similar rejected', -1.0, -1.2, 1.5),
    ('Mediocre chosen, bad rejected', -2.5, -4.0, 2.5),
    ('Both poor quality', -3.5, -4.0, 3.5),
]
for label, lp_c, lp_r, sft in scenarios:
    result = orpo_loss(lp_c, lp_r, sft)
    print(f'{label:<30} {result["sft_loss"]:>7} {result["preference_loss"]:>7} {result["total"]:>7} {result["odds_ratio"]:>11.4f}')

## SimPO: Simple Preference Optimization

SimPO (Meng et al. 2024) addresses a different problem: DPO's implicit reward is the log probability ratio, which is biased toward longer responses because longer sequences naturally have lower log probabilities.

SimPO normalizes by response length and uses a reward margin γ:

```
L_SimPO = -log σ((β/|y_w|) * log π(y_w|x) - (β/|y_l|) * log π(y_l|x) - γ)
```

The length normalization makes the implicit reward length-invariant. The margin γ ensures there is a minimum reward gap between chosen and rejected before the loss contributes meaningfully.

In [ ]:
def simpo_loss(log_prob_chosen: float, log_prob_rejected: float,
               len_chosen: int, len_rejected: int,
               beta: float = 2.0, gamma: float = 0.5) -> float:
    norm_chosen = log_prob_chosen / max(1, len_chosen)
    norm_rejected = log_prob_rejected / max(1, len_rejected)
    logits = beta * (norm_chosen - norm_rejected) - gamma
    return -math.log(1.0 / (1.0 + math.exp(-logits)) + 1e-8)

# Compare DPO and SimPO on length-varied examples
from dataclasses import dataclass

@dataclass
class AlignExample:
    prompt: str
    chosen: str
    rejected: str

examples = [
    ('Short accurate vs long inaccurate',
     -2.0, 10, -8.0, 40),   # short chosen, long rejected
    ('Long accurate vs short inaccurate',
     -8.0, 40, -2.0, 10),   # long chosen, short rejected
    ('Both similar length',
     -3.0, 15, -5.0, 15),
]

print(f'{'Scenario':<35} {'DPO Loss':>10} {'SimPO Loss':>11}')
print('-' * 60)
for label, lp_c, len_c, lp_r, len_r in examples:
    # DPO loss (no ref model: treat ref log probs as 0)
    dpo_logits = 0.1 * (lp_c - lp_r)
    dpo_l = -math.log(1.0/(1.0+math.exp(-dpo_logits)) + 1e-8)
    simpo_l = simpo_loss(lp_c, lp_r, len_c, len_r)
    print(f'{label:<35} {dpo_l:>10.4f} {simpo_l:>11.4f}')

## When to Use Each Method

**DPO**: the safe default. Well-understood behavior, strong empirical results, widely supported in libraries (HuggingFace TRL). Use when you have a reference model already loaded.

**ORPO**: when you want a single training stage that combines SFT and alignment (no separate SFT phase needed). Best for instruction tuning from a base model.

**SimPO**: when you observe verbosity bias in DPO-trained models (longer responses winning regardless of quality). Particularly relevant for conversational tasks where concise answers are preferred.

The field is still converging on which method is definitively best. Most practitioners run DPO as a baseline and experiment with ORPO/SimPO if they observe specific failure modes.